# LCA Lasers4MaaS: R&D validation tests (D13.1)

Unit processes of the validation tests described in D13.1, section 4 (R&D processes):

- **Battery cooling plate** (maquettes and full-size plates): leak tightness test, macrography seam analysis, burst pressure test, pressure cycle (fatigue) test and thermal shock test.
- **Heat exchanger prototype** (leak test chain): cleaning, resistance spot welding (RSW), pickling and electropolishing, and quality test. Laser cutting and forming has no data in D13.1 and is not modelled.

The coupon tests of D13.1 (metallographic analysis and CT scan) are not in this notebook. `L4M_R&D.ipynb` models the CT scan as scanner power x scan time.

Functional unit of each process: one test of one component. D13.1 gives the test procedures (durations, pressures, temperatures, flows) but no power ratings or consumable amounts, so the amounts are **engineering estimates** (`ESTIMATES` below). Each estimate is a min/max range with its source, to be replaced by partner data.

Run `L4M_R&D.ipynb` first with the same background: this notebook copies the inputs of the metallographic steps from the package `Mettalurgic test (CIVAN)` in the project. The processes are written to the database `RD_validation_tests`; `L4M_R&D.ipynb` links them into the prototype rounds (`VALIDATION_TESTS`), so run it again after this notebook.

In [1]:
import os
from pathlib import Path

import bw2data as bd
import pandas as pd
from IPython.display import display

from l4m_lca.constants import BACKGROUNDS, EUROSTAT_ELEC_PRICE_2025S1
from l4m_lca.impacts import BackgroundImpacts
from l4m_lca.lcia import COST_COLUMN, ELECTRICITY_COLUMN, export, scenario_table
from l4m_lca.project import DEFAULT_INDICATORS, ef31_methods, ensure_package, find_activity, method_units, open_project
from l4m_lca.rdtests import (ALUMINIUM_CP_KJ_PER_KG_K, WATER_CP_KJ_PER_KG_K, add, background_inputs_per_unit,
                             burst_test_time_h, cooling_kwh, cycle_time_h, free_air_nm3, heating_kwh, mul,
                             power_time_kwh, rd_test_processes)

## Parameters

In [2]:
# Background database: "ecoinvent" or "bafu". The environment variable L4M_BACKGROUND overrides it.
BACKGROUND = os.environ.get("L4M_BACKGROUND", "ecoinvent")
PROFILE = BACKGROUNDS[BACKGROUND]
PROJECT_NAME = PROFILE["project"]
BACKGROUND_DATABASE = PROFILE["database"]
RESULTS_DIR = PROFILE["results"]

RD_TESTS_NAME = "RD_validation_tests"
METALLURGIC_NAME = "Mettalurgic test (CIVAN)"      # spelling as stored in the package
METALLURGIC_PACKAGE = Path("databases") / "Metallurgic test (CIVAN).bw2package"
# Metallographic preparation steps copied for the macrography seam analysis (package codes)
MACROGRAPHY_STEPS = {
    "04_Cutting":      "7faf722864b94ed797cdd406d22120db",
    "05_Hot-mounting": "06923c5d5a4d416d87c1702173dc648a",
    "06_Grinding":     "f00138e49c5843caa7a1e7d8f33f5986",
    "07_Polishing":    "53c6381979c942e79ed1f043a8b184ed",
    "08_Etching":      "92039648c9a64364ab22f228c0f4b44d",
    "09_Test":         "32bb94122a4542ad8af12fb8ea34a26e",
}

INDICATORS = DEFAULT_INDICATORS
ELECTRICITY_PRICE = EUROSTAT_ELEC_PRICE_2025S1["RER"]   # EUR/kWh

# Background activities: (name, location). A missing dataset is reported with candidates and not modelled.
FLOWS = {
    "ecoinvent": {
        "electricity":       ("market group for electricity, medium voltage", "RER"),
        "water":             ("market group for tap water", "RER"),
        "sodium_hydroxide":  ("market for sodium hydroxide, without water, in 50% solution state", "RER"),  # alkaline detergent proxy
        "nitric_acid":       ("market for nitric acid, without water, in 50% solution state", "RER w/o RU"),
        "hydrofluoric_acid": ("market for hydrogen fluoride", "RER"),
        "phosphoric_acid":   ("market for phosphoric acid, industrial grade, without water, in 85% solution state", "GLO"),
        "sulfuric_acid":     ("market for sulfuric acid", "RER"),
        "acetone":           ("market for acetone, liquid", "RER"),
        "ethylene_glycol":   ("market for ethylene glycol", "RER"),
        "copper":            ("market for copper, cathode", "GLO"),               # RSW electrode wear
    },
    "bafu": {
        "electricity":       ("Electricity, medium voltage, at grid", "RER"),
        "water":             ("Tap water, at user", "RER"),
        "sodium_hydroxide":  ("Sodium hydroxide, 50% in H2O, production mix, at plant", "RER"),
        "nitric_acid":       ("Nitric acid, 50% in H2O, at plant", "RER"),
        "hydrofluoric_acid": ("Hydrogen fluoride, at plant", "GLO"),
        "phosphoric_acid":   ("Phosphoric acid, industrial grade, 85% in H2O, at plant", "RER"),
        "sulfuric_acid":     ("Sulphuric acid, liquid, at plant", "RER"),
        "acetone":           ("Acetone, liquid, at plant", "RER"),
        "ethylene_glycol":   ("Ethylene glycol, at plant", "RER"),
        "copper":            ("Copper, primary, at refinery", "GLO"),
    },
}[BACKGROUND]

In [3]:
# ESTIMATES: {"min", "max"} ranges. Source: "D13.1" = test procedure in D13.1; "estimate" = typical value, to be
# replaced by partner data; "workflow" = PRODUCT_WORKFLOWS in workflow_assembly.ipynb (prototype case).
ESTIMATES = {
    # --- Battery cooling plate (maquette to full-size plate)
    "plate_mass_kg":            {"min": 1.26, "max": 14.38},  # workflow (prototype part weight)
    "plate_volume_l":           {"min": 0.3, "max": 1.5},     # estimate: internal coolant volume of the plate
    "rig_circuit_volume_l":     {"min": 2.0, "max": 5.0},     # estimate: coolant in the test rig (hoses, pump, reservoir)
    "glycol_mass_fraction":     {"min": 0.5, "max": 0.5},     # estimate: 50/50 water-glycol coolant
    "coolant_density_kg_per_l": {"min": 1.07, "max": 1.07},
    "coolant_cp_kj_per_kg_k":   {"min": 3.5, "max": 3.5},
    "compressor_kwh_per_nm3":   {"min": 0.10, "max": 0.12},   # estimate: specific energy of compressed air
    # Leak tightness test
    "leak_cycle_s":             {"min": 99, "max": 99},       # D13.1: pre-fill 25 s ... emptying 4 s
    "leak_fill_gauge_bar":      {"min": 2.5, "max": 2.5},     # D13.1: pre-filling at 2.5 bar (filling at 2 bar)
    "leak_fills":               {"min": 2, "max": 2},         # D13.1: pre-filling and filling
    "leak_tester_kw":           {"min": 0.1, "max": 0.3},     # estimate: pressure-decay leak tester and bath
    "leak_bath_dragout_l":      {"min": 0.1, "max": 0.5},     # estimate: water lost from the after-bath per test
    # Macrography seam analysis
    "macro_specimens":          {"min": 2, "max": 6},         # estimate: cross-sections per plate
    # Burst pressure test
    "burst_pressure_bar":       {"min": 10, "max": 30},       # estimate (D13.1 gives no P_max): ramps at 0.1 bar/s to burst
    "burst_setup_h":            {"min": 0.1, "max": 0.2},     # estimate: filling, connection and venting
    "burst_bench_kw":           {"min": 0.75, "max": 2.2},    # estimate: hydraulic pressure test bench
    "burst_water_loss_l":       {"min": 0.5, "max": 1.0},     # estimate: rig water lost besides the plate volume
    # Pressure cycle (fatigue) test
    "fatigue_cycles":           {"min": 120_000, "max": 120_000},  # D13.1: 2,000 + 98,000 + 20,000 cycles
    "fatigue_frequency_hz":     {"min": 1.0, "max": 1.0},     # D13.1: sinusoidal, 1 Hz
    "fatigue_bench_kw":         {"min": 3.0, "max": 7.5},     # estimate: pulsation unit and coolant conditioning to 23 degC
    # Thermal shock test
    "shock_cycles":             {"min": 10, "max": 10},       # D13.1
    "shock_hot_min":            {"min": 5, "max": 5},         # D13.1: coolant 90 degC, 15 L/min, 1.5 bar
    "shock_cold_min":           {"min": 5, "max": 5},         # D13.1: bath 0 degC
    "shock_hot_c":              {"min": 90, "max": 90},       # D13.1
    "shock_cold_c":             {"min": 0, "max": 0},         # D13.1
    "room_c":                   {"min": 20, "max": 20},
    "heater_efficiency":        {"min": 0.9, "max": 0.95},    # estimate
    "chiller_cop":              {"min": 2.0, "max": 3.5},     # estimate
    "shock_pump_kw":            {"min": 0.2, "max": 0.5},     # estimate: circulation pump at 15 L/min
    "shock_hot_standby_kw":     {"min": 0.5, "max": 1.5},     # estimate: heat loss of the 90 degC loop
    "shock_cold_standby_kw":    {"min": 0.5, "max": 1.5},     # estimate: chiller keeping the 0 degC bath
    # --- Heat exchanger prototype (AISI 316L)
    "hx_area_m2":               {"min": 0.5, "max": 2.0},     # workflow (prototype surface area)
    "hx_volume_l":              {"min": 10, "max": 50},       # estimate: water fill for the quality (leak) test
    # Cleaning (alkaline detergent + acetone, ultrasonic and heating)
    "clean_bath_kw":            {"min": 1.5, "max": 3.0},     # estimate: ultrasonic generator and heater
    "clean_time_h":             {"min": 0.17, "max": 0.33},   # estimate: 10-20 min
    "clean_dragout_l_per_m2":   {"min": 0.05, "max": 0.2},    # estimate
    "clean_detergent_fraction": {"min": 0.03, "max": 0.05},   # estimate: 3-5% alkaline detergent
    "clean_acetone_kg":         {"min": 0.05, "max": 0.2},    # estimate
    "clean_rinse_water_l":      {"min": 10, "max": 30},       # estimate
    # Resistance spot welding
    "rsw_spots":                {"min": 50, "max": 200},      # estimate: spots per heat exchanger
    "rsw_kj_per_spot":          {"min": 5, "max": 20},        # estimate: electricity per spot incl. transformer losses
    "rsw_standby_kw":           {"min": 1.0, "max": 2.0},     # estimate: cooling water and controls
    "rsw_time_h":               {"min": 0.25, "max": 0.5},    # estimate
    "rsw_acetone_kg":           {"min": 0.02, "max": 0.05},   # D13.1 input; amount estimate
    "rsw_electrode_wear_kg":    {"min": 0.0001, "max": 0.0005},  # estimate: copper electrode wear
    # Pickling and electropolishing
    "pickle_dragout_l_per_m2":  {"min": 0.1, "max": 0.2},     # estimate
    "pickle_density_kg_per_l":  {"min": 1.1, "max": 1.1},
    "pickle_hno3_fraction":     {"min": 0.15, "max": 0.20},   # estimate: HNO3/HF pickling solution
    "pickle_hf_fraction":       {"min": 0.02, "max": 0.05},
    "ep_current_a_per_m2":      {"min": 1000, "max": 3000},   # estimate: 10-30 A/dm2
    "ep_voltage_v":             {"min": 6, "max": 12},        # estimate
    "ep_time_h":                {"min": 0.08, "max": 0.25},   # estimate: 5-15 min
    "ep_rectifier_efficiency":  {"min": 0.85, "max": 0.9},
    "ep_dragout_l_per_m2":      {"min": 0.1, "max": 0.2},     # estimate
    "ep_density_kg_per_l":      {"min": 1.7, "max": 1.7},
    "ep_h3po4_fraction":        {"min": 0.50, "max": 0.65},   # estimate: phosphoric/sulfuric acid electrolyte
    "ep_h2so4_fraction":        {"min": 0.15, "max": 0.30},
    "ep_heating_kw":            {"min": 2.0, "max": 6.0},     # estimate: bath at 50-70 degC
    "ep_heating_h":             {"min": 0.1, "max": 0.25},
    "pickle_ep_rinse_l_per_m2": {"min": 20, "max": 50},       # estimate
    # Quality test (water, compressed air, acetone, videoendoscope and altimeter)
    "quality_air_nm3":          {"min": 0.05, "max": 0.2},    # estimate
    "quality_instruments_kw":   {"min": 0.05, "max": 0.1},    # estimate
    "quality_time_h":           {"min": 0.5, "max": 1.0},     # estimate
    "quality_acetone_kg":       {"min": 0.05, "max": 0.1},    # estimate
}
E = ESTIMATES

## Project and background data

In [4]:
background = open_project(PROJECT_NAME, BACKGROUND_DATABASE)
print(f"Background: {BACKGROUND_DATABASE} (project {PROJECT_NAME}), results in {RESULTS_DIR}/")

def lookup(name, spec):
    """Background activity, or None with a list of candidates when the dataset is not found."""
    try:
        return find_activity(BACKGROUND_DATABASE, *spec)
    except Exception:
        words = spec[0].replace("market group for ", "").replace("market for ", "").split(",")[0]
        hits = bd.Database(BACKGROUND_DATABASE).search(words, limit=8)
        print(f"NOT FOUND {name}: {spec}. Candidates: {[(a['name'], a['location']) for a in hits]}")
        return None

flows = {name: lookup(name, spec) for name, spec in FLOWS.items()}
assert flows["electricity"] is not None, "The electricity dataset is required"

if METALLURGIC_NAME not in bd.databases:
    assert BACKGROUND == "ecoinvent", "Run L4M_R&D.ipynb with BACKGROUND = 'bafu' first: it relinks the package"
    ensure_package(METALLURGIC_NAME, METALLURGIC_PACKAGE)
metallurgic = {label: bd.Database(METALLURGIC_NAME).get(code) for label, code in MACROGRAPHY_STEPS.items()}

methods = ef31_methods(INDICATORS)
units = method_units(methods)

Background: ecoinvent-3.12-cutoff (project Lasers4MaaS), results in results/


NOT FOUND hydrofluoric_acid: ('market for hydrogen fluoride', 'GLO'). Candidates: [('hydrogen fluoride production', 'RoW'), ('market for hydrogen fluoride', 'RoW'), ('hydrogen fluoride production', 'RER'), ('vinyl fluoride production', 'RoW'), ('vinyl fluoride production', 'US'), ('market for hydrogen fluoride', 'RER'), ('sodium fluoride production', 'GLO'), ('hexafluoroethane production, from fluorination of tetrafluoroethane', 'GLO')]
NOT FOUND ethylene_glycol: ('market for ethylene glycol', 'GLO'). Candidates: [('ethylene glycol monoethyl ether production', 'RER'), ('market for ethylene glycol', 'RER'), ('ethylene glycol diethyl ether production', 'RoW'), ('market for ethylene glycol monoethyl ether', 'RER'), ('ethylene glycol monoethyl ether production', 'RoW'), ('market for ethylene glycol diethyl ether', 'GLO'), ('market for ethylene glycol monoethyl ether', 'RoW'), ('market for ethylene glycol dimethyl ether', 'GLO')]


## Unit processes of the tests

Electricity of each test = equipment power x duration (durations from D13.1), plus heating and cooling
(mass x heat capacity x temperature difference) and compressed air (free air x specific energy).
The macrography copies the direct inputs of the metallographic preparation steps of the CIVAN package,
per specimen, without their upstream chain (the welded coupon).

In [5]:
air = lambda nm3: mul(nm3, E["compressor_kwh_per_nm3"])
coolant_kg = mul(E["rig_circuit_volume_l"], E["coolant_density_kg_per_l"])
glycol_kg = mul(coolant_kg, E["glycol_mass_fraction"])
leak_key = (RD_TESTS_NAME, "leak_tightness_test")

# Leak tightness test: tester running for one cycle + compressed air for pre-filling and filling
leak_air = mul(free_air_nm3(E["plate_volume_l"], E["leak_fill_gauge_bar"]), E["leak_fills"])
leak_kwh = add(power_time_kwh(E["leak_tester_kw"], mul(E["leak_cycle_s"], 1 / 3600)), air(leak_air))

# Burst test: bench running for the ramps, the two 5-min holds and the set-up; the plate's water is discarded
burst_kwh = power_time_kwh(E["burst_bench_kw"], add(burst_test_time_h(E["burst_pressure_bar"]), E["burst_setup_h"]))
burst_water_kg = add(E["plate_volume_l"], E["burst_water_loss_l"])

# Pressure cycle test: 120,000 cycles at 1 Hz (33.3 h), then a leak tightness test
fatigue_kwh = power_time_kwh(E["fatigue_bench_kw"], cycle_time_h(E["fatigue_cycles"], E["fatigue_frequency_hz"]))

# Thermal shock: per cycle the plate is heated to 90 degC and cooled to 0 degC; pumps and standby during all cycles;
# the coolant is heated once from room temperature to 90 degC. Then a leak tightness test.
shock_h = mul(E["shock_cycles"], add(E["shock_hot_min"], E["shock_cold_min"]), 1 / 60)
plate_dt = (E["shock_hot_c"]["min"] - E["shock_cold_c"]["max"], E["shock_hot_c"]["max"] - E["shock_cold_c"]["min"])
shock_kwh = add(
    mul(E["shock_cycles"], heating_kwh(E["plate_mass_kg"], ALUMINIUM_CP_KJ_PER_KG_K, plate_dt, E["heater_efficiency"])),
    mul(E["shock_cycles"], cooling_kwh(E["plate_mass_kg"], ALUMINIUM_CP_KJ_PER_KG_K, plate_dt, E["chiller_cop"])),
    heating_kwh(coolant_kg, E["coolant_cp_kj_per_kg_k"],
                (E["shock_hot_c"]["min"] - E["room_c"]["max"], E["shock_hot_c"]["max"] - E["room_c"]["min"]),
                E["heater_efficiency"]),
    power_time_kwh(add(E["shock_pump_kw"], E["shock_hot_standby_kw"], E["shock_cold_standby_kw"]), shock_h),
)

# Heat exchanger: cleaning, RSW, pickling + electropolishing, quality test
area = E["hx_area_m2"]
clean_kwh = power_time_kwh(E["clean_bath_kw"], E["clean_time_h"])
clean_naoh = mul(area, E["clean_dragout_l_per_m2"], E["clean_detergent_fraction"])          # kg (density 1)
rsw_kwh = add(mul(E["rsw_spots"], E["rsw_kj_per_spot"], 1 / 3600), power_time_kwh(E["rsw_standby_kw"], E["rsw_time_h"]))
pickle_kg = mul(area, E["pickle_dragout_l_per_m2"], E["pickle_density_kg_per_l"])
ep_kg = mul(area, E["ep_dragout_l_per_m2"], E["ep_density_kg_per_l"])
ep_kwh = add(
    (mul(area, E["ep_current_a_per_m2"], E["ep_voltage_v"], E["ep_time_h"], 1 / 1000)[0] / E["ep_rectifier_efficiency"]["max"],
     mul(area, E["ep_current_a_per_m2"], E["ep_voltage_v"], E["ep_time_h"], 1 / 1000)[1] / E["ep_rectifier_efficiency"]["min"]),
    power_time_kwh(E["ep_heating_kw"], E["ep_heating_h"]),
)
quality_kwh = add(power_time_kwh(E["quality_instruments_kw"], E["quality_time_h"]), air(E["quality_air_nm3"]))

# Macrography: direct inputs of the metallographic steps per specimen, times the number of specimens
copied = [(node, kind, amount, E["macro_specimens"])
          for step in metallurgic.values()
          for node, kind, amount in background_inputs_per_unit(step, {METALLURGIC_NAME})]

SPECS = {
    "leak_tightness_test": {
        "name": "Leak tightness test, battery cooling plate", "unit": "test",
        "comment": "D13.1: cyclic pressurisation under water, 99 s. Electricity of the tester and compressed air.",
        "electricity_kwh": leak_kwh, "inputs": {"water": E["leak_bath_dragout_l"]}},
    "macrography_seam_analysis": {
        "name": "Macrography seam analysis, battery cooling plate", "unit": "test",
        "comment": "D13.1: cutting, mounting, grinding, polishing, etching and examination of 2-6 cross-sections.",
        "copied": copied},
    "burst_pressure_test": {
        "name": "Burst pressure test, battery cooling plate", "unit": "test",
        "comment": "D13.1: ramps at 0.1 bar/s, holds of 5 min at Pmax and 2 Pmax, until burst. The plate is destroyed.",
        "electricity_kwh": burst_kwh, "inputs": {"water": burst_water_kg}},
    "pressure_cycle_test": {
        "name": "Pressure cycle (fatigue) test, battery cooling plate", "unit": "test",
        "comment": "D13.1: 120,000 cycles at 1 Hz, coolant back to 23 degC, followed by a leak tightness test.",
        "electricity_kwh": fatigue_kwh, "inputs": {"ethylene_glycol": glycol_kg, "water": mul(coolant_kg, add(1, -E["glycol_mass_fraction"]["min"]))},
        "links": [(leak_key, 1)]},
    "thermal_shock_test": {
        "name": "Thermal shock test, battery cooling plate", "unit": "test",
        "comment": "D13.1: 10 cycles of 5 min coolant at 90 degC and 5 min immersion at 0 degC, then a leak tightness test.",
        "electricity_kwh": shock_kwh, "inputs": {"ethylene_glycol": glycol_kg, "water": mul(coolant_kg, add(1, -E["glycol_mass_fraction"]["min"]))},
        "links": [(leak_key, 1)]},
    "hx_cleaning": {
        "name": "Cleaning, heat exchanger prototype", "unit": "test",
        "comment": "D13.1: alkaline detergent and acetone, ultrasonic agitation and heating.",
        "electricity_kwh": clean_kwh,
        "inputs": {"sodium_hydroxide": clean_naoh, "acetone": E["clean_acetone_kg"], "water": E["clean_rinse_water_l"]}},
    "hx_resistance_spot_welding": {
        "name": "Resistance spot welding, heat exchanger prototype", "unit": "test",
        "comment": "D13.1: RSW of AISI 316L sheets and spacers before laser welding.",
        "electricity_kwh": rsw_kwh, "inputs": {"acetone": E["rsw_acetone_kg"], "copper": E["rsw_electrode_wear_kg"]}},
    "hx_pickling_electropolishing": {
        "name": "Pickling and electropolishing, heat exchanger prototype", "unit": "test",
        "comment": "D13.1: pickling solution, electropolishing solution, water, electricity for heating and electropolishing.",
        "electricity_kwh": ep_kwh,
        "inputs": {"nitric_acid": mul(pickle_kg, E["pickle_hno3_fraction"]),
                   "hydrofluoric_acid": mul(pickle_kg, E["pickle_hf_fraction"]),
                   "phosphoric_acid": mul(ep_kg, E["ep_h3po4_fraction"]),
                   "sulfuric_acid": mul(ep_kg, E["ep_h2so4_fraction"]),
                   "water": mul(area, E["pickle_ep_rinse_l_per_m2"])}},
    "hx_quality_test": {
        "name": "Quality test (leak test), heat exchanger prototype", "unit": "test",
        "comment": "D13.1: water, compressed air, acetone, electricity for the videoendoscope and altimeter.",
        "electricity_kwh": quality_kwh, "inputs": {"water": E["hx_volume_l"], "acetone": E["quality_acetone_kg"]}},
}

data, ignored = rd_test_processes(RD_TESTS_NAME, SPECS, flows)
for code_, names in ignored.items():
    print(f"{code_}: not modelled (no background dataset): {names}")
rd_tests = bd.Database(RD_TESTS_NAME)
rd_tests.write(data)
pd.DataFrame([{"Process": d["name"], "Inputs": len(d["exchanges"]) - 1} for d in data.values()])

pressure_cycle_test: not modelled (no background dataset): ['ethylene_glycol']
thermal_shock_test: not modelled (no background dataset): ['ethylene_glycol']
hx_pickling_electropolishing: not modelled (no background dataset): ['hydrofluoric_acid']
15:02:31+0100 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


  0%|          | 0/9 [00:00<?, ?it/s]

100%|██████████| 9/9 [00:00<00:00, 17181.95it/s]

15:02:31+0100 [info     ] Vacuuming database            


,Process,Inputs
0,"Leak tightness test, battery cooling plate",2
1,"Macrography seam analysis, battery cooling plate",24
2,"Burst pressure test, battery cooling plate",2
3,"Pressure cycle (fatigue) test, battery cooling...",3
4,"Thermal shock test, battery cooling plate",3
5,"Cleaning, heat exchanger prototype",4
6,"Resistance spot welding, heat exchanger prototype",3
7,"Pickling and electropolishing, heat exchanger ...",5
8,"Quality test (leak test), heat exchanger proto...",3


## LCIA results

In [6]:
impacts = BackgroundImpacts(methods, flows["electricity"])
lcia_df = scenario_table([rd_tests], impacts, {flows["electricity"].key: ELECTRICITY_PRICE})
export(lcia_df, units, RESULTS_DIR, RD_TESTS_NAME)

GHG = next(c for c in lcia_df.columns if c.startswith("climate change"))
lcia_df.pivot_table(index="Process", columns="Scenario", values=[GHG, ELECTRICITY_COLUMN], sort=False).round(3)

/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 3.43e+12)
  warnings.warn(msg, UmfpackWarning)


Results saved to results/RD_validation_tests_LCIA_results.csv and results/RD_validation_tests_LCIA_results.json


climate change: total (EF v3.1 - IPCC 2021)  \
Scenario                                                                               minimum   
Process                                                                                          
Burst pressure test, battery cooling plate                                               0.080   
Cleaning, heat exchanger prototype                                                       0.215   
Pickling and electropolishing, heat exchanger p...                                       0.240   
Quality test (leak test), heat exchanger prototype                                       0.134   
Resistance spot welding, heat exchanger prototype                                        0.164   
Leak tightness test, battery cooling plate                                               0.001   
Macrography seam analysis, battery cooling plate                                         0.969   
Pressure cycle (fatigue) test, battery cooling ...                                      36.016   
Thermal shock test, battery cooling plate                                                0.914   

                                                                    \
Scenario                                            amount maximum   
Process                                                              
Burst pressure test, battery cooling plate           0.219   0.357   
Cleaning, heat exchanger prototype                   0.540   0.865   
Pickling and electropolishing, heat exchanger p...   4.583   8.926   
Quality test (leak test), heat exchanger prototype   0.217   0.300   
Resistance spot welding, heat exchanger prototype    0.523   0.883   
Leak tightness test, battery cooling plate           0.002   0.004   
Macrography seam analysis, battery cooling plate     1.939   2.908   
Pressure cycle (fatigue) test, battery cooling ...  63.029  90.041   
Thermal shock test, battery cooling plate            2.521   4.128   

                                                   Electricity Consumption (kWh)  \
Scenario                                                                 minimum   
Process                                                                            
Burst pressure test, battery cooling plate                                 0.221   
Cleaning, heat exchanger prototype                                         0.255   
Pickling and electropolishing, heat exchanger p...                         0.467   
Quality test (leak test), heat exchanger prototype                         0.030   
Resistance spot welding, heat exchanger prototype                          0.319   
Leak tightness test, battery cooling plate                                 0.003   
Macrography seam analysis, battery cooling plate                           1.578   
Pressure cycle (fatigue) test, battery cooling ...                       100.000   
Thermal shock test, battery cooling plate                                  2.533   

                                                                      
Scenario                                             amount  maximum  
Process                                                               
Burst pressure test, battery cooling plate            0.605    0.990  
Cleaning, heat exchanger prototype                    0.622    0.990  
Pickling and electropolishing, heat exchanger p...   11.572   22.676  
Quality test (leak test), heat exchanger prototype    0.077    0.124  
Resistance spot welding, heat exchanger prototype     1.215    2.111  
Leak tightness test, battery cooling plate            0.006    0.009  
Macrography seam analysis, battery cooling plate      3.156    4.735  
Pressure cycle (fatigue) test, battery cooling ...  175.000  250.000  
Thermal shock test, battery cooling plate             6.992   11.451